# Agente Briefing

### Grupo 1

- Amarildo Lucena
- Cassiane Bueno
- Flavia Danzi
- Israel Siqueira
- Rafael Winter

## Descrição

Este notebook cria um agente que produz um briefing para o usuário contendo informações como a localização atual, baseada em IP, notícias, clima, cotação do dólar e uma frase motivacional.

### API Usadas

- https://gnews.io/ como API de noticias
- https://openweathermap.org/api como API de clima
- https://api.exchangerate.host/convert como API de cotação do dólar
- https://api.adviceslip.com/advice como API de mensagem motivacional


In [71]:
import os
import sys
from pathlib import Path
from pprint import pprint
import requests
from typing import Dict, List, Any, Optional

from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain.messages import HumanMessage, ToolMessage

# Import utils from books directory
sys.path.insert(0, str(Path.cwd().parent / 'books'))
from utils import format_messages, show_prompt, console

load_dotenv()

True

## Funções auxiliares

In [72]:
def get_location() -> Dict[str, Any]:
    """Infer user location by IP using ipinfo.io (public, no key required for basic info).

    Returns: dict with city, region, country, loc (lat,lon)
    """
    try:
        r = requests.get("https://ipinfo.io/json", timeout=5)
        r.raise_for_status()
        data = r.json()
        loc = data.get("loc", "")
        lat, lon = (loc.split(",") if loc else (None, None))
        return {
            "city": data.get("city"),
            "region": data.get("region"),
            "country": data.get("country"),
            "loc": {"lat": lat, "lon": lon},
        }
    except Exception as e:
        return {"city": None, "region": None, "country": None, "loc": {"lat": None, "lon": None}, "error": str(e)}


def fetch_news(gnews_key: Optional[str], country: Optional[str] = None, max_results: int = 3) -> List[Dict[str, Any]]:
    """Fetch top headlines from gnews.io. Expects API key in `gnews_key`.

    Returns list of {title, description, url}
    """
    if not gnews_key:
        return [{"title": "API key for GNews not provided", "description": "", "url": ""}]
    params = {
        "token": gnews_key,
        "lang": "pt",
        "max": max_results,
    }
    if country:
        params["country"] = country.lower()
    try:
        r = requests.get("https://gnews.io/api/v4/top-headlines", params=params, timeout=7)
        r.raise_for_status()
        data = r.json()
        articles = data.get("articles", [])
        results = []
        for a in articles[:max_results]:
            results.append({
                "title": a.get("title"),
                "description": a.get("description"),
                "url": a.get("url"),
            })
        return results
    except Exception as e:
        return [{"title": "Erro ao buscar notícias", "description": str(e), "url": ""}]


def fetch_weather(owm_key: Optional[str], lat: Optional[str], lon: Optional[str]) -> Dict[str, Any]:
    """Fetch current weather from OpenWeatherMap using lat/lon. Returns summary dict."""
    if not owm_key or not lat or not lon:
        return {"error": "missing openweathermap key or coordinates"}
    try:
        params = {"lat": lat, "lon": lon, "units": "metric", "appid": owm_key, "lang": "pt"}
        r = requests.get("https://api.openweathermap.org/data/2.5/weather", params=params, timeout=7)
        r.raise_for_status()
        data = r.json()
        main = data.get("main", {})
        weather = data.get("weather", [{}])[0]
        return {
            "temp": main.get("temp"),
            "feels_like": main.get("feels_like"),
            "description": weather.get("description"),
            "raw": data,
        }
    except Exception as e:
        return {"error": str(e)}

COUNTRY_TO_CURRENCY = {
    "BR": "BRL",
    "US": "USD",
    "PT": "EUR",
    "DE": "EUR",
    "FR": "EUR",
    "IN": "INR",
    "GB": "GBP",
    "CA": "CAD",
    # add more as needed
}

def fetch_usd_rate(api_key: Optional[str], target_currency: Optional[str]) -> Dict[str, Any]:
    """Fetch USD -> target_currency conversion using exchangerate.host"""
    if not target_currency:
        return {"error": "target currency missing"}
    if not api_key:
        return {"error": "API key missing"}
    try:
        params = {
            "access_key": api_key,
            "from": "USD",
            "to": target_currency,
            "amount": 1,
            "format": 1
        }
        r = requests.get("http://api.exchangerate.host/convert", params=params, timeout=5)
        r.raise_for_status()
        data = r.json()
        
        # Check if the API returned success
        if not data.get("success", False):
            error_info = data.get("error", {})
            return {"error": f"{error_info.get('type', 'unknown')}: {error_info.get('info', 'API error')}"}
        
        return {"rate": data.get("result"), "info": data}
    except Exception as e:
        return {"error": str(e)}


def fetch_advice() -> str:
    try:
        r = requests.get("https://api.adviceslip.com/advice", timeout=4)
        r.raise_for_status()
        data = r.json()
        return data.get("slip", {}).get("advice", "")
    except Exception:
        return "Mantenha o foco e siga em frente."

## Criação de `Tools` do LangChain

In [73]:
@tool
def _tool_location() -> str:
    """Infer user location by IP using ipinfo.io (public, no key required for basic info).

    Returns: dict with city, region, country, loc (lat,lon)
    """
    loc = get_location()
    city = loc.get("city")
    region = loc.get("region")
    country = loc.get("country")
    return f"{city or ''}, {region or ''}, {country or ''} | loc={loc.get('loc')}"

@tool
def _tool_news(input_text: str) -> str:
    """Fetch top headlines from gnews.io. Expects API key in `gnews_key`.

    Returns list of {title, description, url}
    """
    key = os.getenv("GNEWS_API_KEY")
    # input may contain a country code or city; try to use country if provided
    country = None
    if input_text and len(input_text) == 2:
        country = input_text
    articles = fetch_news(key, country=country)
    out = []
    for a in articles:
        out.append(f"- {a.get('title')}: {a.get('description')}")
    return "\n".join(out)

@tool
def _tool_weather(input_text: str) -> str:
    """Fetch current weather from OpenWeatherMap using lat/lon. Returns summary dict."""
    key = os.getenv("OPENWEATHER_API_KEY")
    # input_text expected to be 'lat,lon'
    if not input_text:
        return "Coordinates missing"
    lat, lon = [p.strip() for p in input_text.split(",")]
    w = fetch_weather(key, lat, lon)
    if "error" in w:
        return f"Erro: {w['error']}"
    return f"{w.get('temp')}°C, {w.get('description')} (sensação {w.get('feels_like')}°C)"

@tool
def _tool_exchange(input_text: str) -> str:
    """Fetch USD -> target_currency conversion using exchangerate.host
    
    Args:
        input_text: Currency code (e.g., 'BRL', 'EUR') or currency pair (e.g., 'USD/BRL')
    """
    key = os.getenv("EXCHANGEHOST_KEY")
    
    # Parse input - handle both "BRL" and "USD/BRL" formats
    if not input_text:
        target = "BRL"
    else:
        input_clean = input_text.strip().upper()
        # If it's a currency pair like "USD/BRL", extract the target currency
        if "/" in input_clean:
            parts = input_clean.split("/")
            target = parts[-1]  # Get the target currency (after the /)
        else:
            target = input_clean
    
    r = fetch_usd_rate(key, target)
    if "error" in r:
        return f"Erro: {r['error']}"
    return f"1 USD = {r.get('rate')} {target}"

@tool
def _tool_advice() -> str:
    """Fetch a motivational advice phrase."""
    return fetch_advice()

## Definição do Agente

In [74]:
def create_agent() -> ChatOpenAI:
    """
    Create a lightweight agent object compatible with LangChain v1.0.1 usage in this script.

    Instead of using the higher-level `initialize_agent` helper (which may vary
    between LangChain versions), we return the LLM instance and a mapping of tool
    callables. This keeps the code explicit and compatible with v1+.
    """
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)

    with_tools = llm.bind_tools([_tool_location, _tool_advice, _tool_exchange, _tool_news, _tool_weather])

    return with_tools

## Função de Briefing

In [75]:
def build_briefing() -> dict[str, Any]:
    """Runs the agent to produce a briefing in Portuguese, casual and objective.

    It will: infer location, fetch news for the country, weather for coordinates, 
    USD rate to local currency, and an advice.
    """
    # Mapping of tool names to their callable functions
    tool_map = {
        "_tool_location": _tool_location,
        "_tool_news": _tool_news,
        "_tool_weather": _tool_weather,
        "_tool_exchange": _tool_exchange,
        "_tool_advice": _tool_advice,
    }

    # Build a prompt that instructs the agent to use the provided tools
    system_prompt = (
        "Você é um assistente que produz um briefing curto e objetivo em português (PT-BR).\n"
        "Use as ferramentas disponíveis para:\n"
        "1) inferir a minha localização;\n"
        "2) obter notícias recentes relevantes para o país/município que estou;\n"
        "3) obter o clima atual a partir das coordenadas;\n"
        "4) obter a cotação atual do dólar na moeda local;\n"
        "5) finalizar com uma frase motivacional.\n"
        "Responda de forma casual, direta e em poucas frases.\n"
    )
    
    agent = create_agent()
    user_prompt = "olá, me atualize"
    full_prompt = system_prompt + "\n" + user_prompt
    
    # Display the initial prompt using utils
    show_prompt(system_prompt, title="System Prompt")
    show_prompt(user_prompt, title="User Prompt")

    # Initial conversation messages
    messages = [HumanMessage(content=full_prompt)]
    
    # Maximum iterations to prevent infinite loops
    max_iterations = 10
    iteration = 0
    
    while iteration < max_iterations:
        iteration += 1
        result = agent.invoke(messages)
        
        # Check if the agent wants to call any tools
        if not hasattr(result, "tool_calls") or not result.tool_calls:
            # No more tool calls - agent has produced the final response
            messages.append(result)
            
            # Display full conversation using utils
            format_messages(messages)
            
            return result.to_json()
        
        # Add AI message to conversation
        messages.append(result)
        
        # Process all tool calls
        for call in result.tool_calls:
            tool_name = call['name']
            args = call['args']
            tool_call_id = call['id']
            
            # Get the appropriate tool function
            tool_func = tool_map.get(tool_name)
            if not tool_func:
                error_msg = f"Unknown tool: {tool_name}"
                tool_message = ToolMessage(
                    name=tool_name,
                    content=error_msg,
                    tool_call_id=tool_call_id
                )
            else:
                # Invoke the tool with its arguments
                try:
                    tool_output = tool_func.invoke(args)
                    tool_message = ToolMessage(
                        name=tool_name,
                        content=str(tool_output),
                        tool_call_id=tool_call_id
                    )
                except Exception as e:
                    error_msg = f"Error executing {tool_name}: {str(e)}"
                    tool_message = ToolMessage(
                        name=tool_name,
                        content=error_msg,
                        tool_call_id=tool_call_id
                    )
            
            messages.append(tool_message)
    
    # If we've exhausted iterations, display conversation anyway
    format_messages(messages)
    return result.to_json()

## Executa o Briefing

In [76]:
# Run the agent to generate briefing
try:
    result = build_briefing()
except Exception as e:
    print(f"Erro ao gerar briefing: {e}")

╭───────────────────────────────────────────────── System Prompt ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Você é um assistente que produz um briefing curto e objetivo em português (PT-BR).                             │
│  Use as ferramentas disponíveis para:                                                                           │
│  1) inferir a minha localização;                                                                                │
│  2) obter notícias recentes relevantes para o país/município que estou;                                         │
│  3) obter o clima atual a partir das coordenadas;                                                               │
│  4) obter a cotação atual do dólar na moeda local;                                                              │
│  5) finalizar com uma frase motivacional.                                                                       │
│  Responda de forma casual, direta e em poucas frases.                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── User Prompt ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  olá, me atualize                                                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────── 🧑 Human ────────────────────────────────────────────────────╮
│ Você é um assistente que produz um briefing curto e objetivo em português (PT-BR).                              │
│ Use as ferramentas disponíveis para:                                                                            │
│ 1) inferir a minha localização;                                                                                 │
│ 2) obter notícias recentes relevantes para o país/município que estou;                                          │
│ 3) obter o clima atual a partir das coordenadas;                                                                │
│ 4) obter a cotação atual do dólar na moeda local;                                                               │
│ 5) finalizar com uma frase motivacional.                                                                        │
│ Responda de forma casual, direta e em poucas frases.                                                            │
│                                                                                                                 │
│ olá, me atualize                                                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────────── 📝 AI ─────────────────────────────────────────────────────╮
│                                                                                                                 │
│                                                                                                                 │
│ 🔧 Tool Call: _tool_location                                                                                    │
│    Args: {}                                                                                                     │
│    ID: call_oyfWJf4XdkaUWqdIsKc5VWfy                                                                            │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🔧 Tool Output ─────────────────────────────────────────────────╮
│ Recife, Pernambuco, BR | loc={'lat': '-8.0539', 'lon': '-34.8811'}                                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭──────────────────────────────────────────────── 🔧 Tool Output ─────────────────────────────────────────────────╮
│ - Buscas por cometa sobem após comportamento incomum do 3I/ATLAS: Corpo celeste de fora do nosso Sistema Solar  │
│ tem gerado curiosidade durante sua passagem                                                                     │
│ - PJ investiga suspeitas de ilegalidades na venda de ativos imobiliários do Novobanco com prejuízos para o      │
│ Estado: A Polícia Judiciária está a realizar buscas esta quarta-feira nas instalações da consultora KPMG.       │
│ Também a sede do Novobanco está a ser alvo de buscas.                                                           │
│ - MP 1.304 prevê quatro níveis de armazenamento para o sistema elétrico, diz Braga: Senador também disse à      │
│ imprensa que os direitos adquiridos pela Lei 14.300/2022 foram preservados no relatório final da MP 1.304.      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🔧 Tool Output ─────────────────────────────────────────────────╮
│ 26.02°C, nuvens dispersas (sensação 26.02°C)                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🔧 Tool Output ─────────────────────────────────────────────────╮
│ 1 USD = 5.359025 BRL                                                                                            │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🔧 Tool Output ─────────────────────────────────────────────────╮
│ Most things are not as bad as you think they are.                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────────── 📝 AI ─────────────────────────────────────────────────────╮
│ E aí! Aqui estão as atualizações pra você:                                                                      │
│                                                                                                                 │
│ 🌍 **Localização**: Recife, Pernambuco.                                                                         │
│                                                                                                                 │
│ 📰 **Notícias Recentes**:                                                                                       │
│ - Buscas por cometa sobem após comportamento incomum do 3I/ATLAS.                                               │
│ - PJ investiga suspeitas de ilegalidades na venda de ativos do Novobanco.                                       │
│ - MP 1.304 prevê quatro níveis de armazenamento para o sistema elétrico.                                        │
│                                                                                                                 │
│ 🌤️ **Clima**: Está fazendo 26.02°C com nuvens dispersas.                                                         │
│                                                                                                                 │
│ 💵 **Cotação do Dólar**: 1 USD = 5.36 BRL.                                                                      │
│                                                                                                                 │
│ 💪 **Frase Motivacional**: "A maioria das coisas não é tão ruim quanto você pensa que são."                     │
│                                                                                                                 │
│ Se precisar de mais alguma coisa, é só avisar!                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯